# Machine-learning for yield predictions

## Objectives {.unnumbered}

-   Review fundamentals of machine-learning methods used in crop science and ecology.

-   Compare and contrast machine-learning methods with process-based modeling and traditional statistical modeling (e.g., generalized linear models).

-   Apply machine-learning methods to predict crop yield in response to environmental and management variables.


## Readings {.unnumbered}

1. (required) [Rosbach (2018) Blog: Neural networks vs. random forests -- does it always have to be deep learning?](https://blog.frankfurt-school.de/neural-networks-vs-random-forests-does-it-always-have-to-be-deep-learning/)

2. (required) [Ellis P (2016) Extrapolation is tough for trees!, Free Range Statistics Blog](https://freerangestats.info/blog/2016/12/10/extrapolation)

3. (recommended) [Jeong et al. (2016) Random forests for global and regional crop yield predictions. PLoS One, 11: e0156571](https://doi.org/10.1371/journal.pone.0156571)

4. (optional) [Zhang et al. (2023) Machine learning versus crop growth models: An ally, not a rival. AoB Plants 15(2): plac061](https://doi.org/10.1093/aobpla/plac061)


### Further readings and additional resources {.unnumbered}

1. @Lucas2020 [A translucent box: interpretable machine learning in ecology. Ecological Monographs 90(4), e01422](https://doi.org/10.1002/ecm.1422)

2. @Borowiec2022 [Deep learning as a tool for ecology and evolution. Methods in Ecology and Evolution 13(8), 1640-1660](https://doi.org/10.1111/2041-210X.13901)

3. @Christin2019 [Applications for deep learning in ecology. Methods in Ecology and Evolution 10(10), 1632-1644](https://besjournals.onlinelibrary.wiley.com/doi/abs/10.1111/2041-210X.13256)

4. [Benchmarking of RF and NN](https://mljar.com/machine-learning/neural-network-vs-random-forest/)


## Questions {.unnumbered}

1.  Give an example of machine-learning (ML) applications in your field of research/work and describe the ML method used.

3.  How does a machine-learning model differ from 1) a mechanistic, process-based model, and 2) a conventional statistical model such as generalized linear model?

3.  Describe Random Forests (RF) and Deep Neural Network (DNN) modeling. Examine and discuss the differences shown with [out-of-sample extrapolation figure](https://freerangestats.info/blog/2016/12/10/extrapolation). 

4.  Can or should machine-learning models replace the roles of process-based models in biology, ecology and other related fields? Based on readings and your own thoughts, discuss ways in which these modeling approaches could work together. Provide one or more examples.

### Terminology {.unnumbered}

machine-learning (ML), decision tree, classification and regression trees (CART), random forests (RF), neural networks (NN), deep learning, artificial intelligence (AI), extreme graident boosting (xgBoost)

## Introduction

In this case study, we will examine some of the statistical and machine-learning models commonly used as regressor for predicting crop yield and other continuous quantities. Exercises will be using data from [Jeong et al. (2016)](https://doi.org/10.1371/journal.pone.0156571). Specifically, we will use data from this paper for potato yield from the Northeast Seaboard Region (NESR) ([Resop, Fleisher, Timlin, & Reddy, 2014](https://doi.org/10.2134/agronj2013.0277)). Corn silage data from the same region are also available for homework exercises. Jeong et al. (2016) evaluated multiple linear regression and random forests regression. Here, we will add a neural network method using [`MLJ.jl`](https://github.com/JuliaAI/MLJ.jl) package. 

## Exercises

In this lab, we will use MLJ package in Julia for machine learning (alternatively you can use a R code with RStudio) to apply statistical and machine learning methods for crop yield predictions using the data from [@Jeong2016]. Specifically, we will use data from this paper for potato yield from the Northeast Seaboard Region (NESR) [@Resop2014]. Corn silage data from the same region are also available for homework exercises. [@Jeong2016] evaluated multiple linear regression and random forests regression. Here, we will add a DNN method using TensorFlow package for comparison.\

In [ ]:
import Pkg

In [ ]:
using Logging
Logging.disable_logging(Logging.Warn)
Logging.disable_logging(Logging.Info)

In [ ]:
using MLJ

In [ ]:
using CSV
using DataFrames
using DataFramesMeta: DataFramesMeta, @subset
using Plots

We will also use [Flux.jl](https://github.com/FluxML/Flux.jl) later in the notebook. It turns out that we have to import it early, otherwise our JupyterHub would crash presumably due to memory limitation.

In [ ]:
using Flux

### Data prep

:::{#exr-ML-data}
### Data preparation
We will import and process data used in [@Jeong2016] to train and test three regression methods: generalized linear model(GLM), random forests (RF), and deep neural networks (DNN) to predict potato yield from NESR. These data were compiled from multiple sources including county yield statistics, weather data, soil database (SSURGO), and geospatial data.

1.  Import data \"NESR-potato.csv\" into a dataframe

2.  Remove observations with missing or 0 values in \"obs_yield\" column

3.  Randomly select 70% of data for model training and save the rest for model testing.\
:::

#### Ex 1.1

Import data "NESR-potato.csv" into a dataframe.

In [ ]:
crop = CSV.read("./data/NESR-potato.csv", DataFrame);

#### Ex 1.2

Remove observations with missing or 0 values in "obs_yield" column.

In [ ]:
crop = @subset(crop, :obs_yield .> 2) |> dropmissing;

#### Ex 1.3

Randomly select 70% of data for model training and save the rest for model testing.

Here we exclude `water` column since the value looks categorical. We assume continuous values for our regression models.

In [ ]:
y, X = unpack(crop, ==(:obs_yield), !=(:water), c -> true);

In [ ]:
train, test = partition(eachindex(y), 0.7)

For the structure of our dataset, we can choose many different types of machine learning models.

In [ ]:
models(matching(X, y))

### Linear Regression

::: {#exr-ML-GLM}
### Generalized Linear Model (GLM)
We will build a multiple linear regression model to predict crop yield based on environmental and management data as input. Use \"lm\" package for generalized linear models (GLM) in R.

1.  Train GLM to predict crop yield in the training data.

2.  Plot predicted vs. observed yield of training data with a regression line and 1:1 line embedded.

3.  Predict crop yield in the test data based the GLM model you just
    trained

4.  Plot predicted vs. observed yield of test data with a regression
    line and 1:1 line embedded.

5.  Discuss and interpret your results\
:::

R: Using "lm" package for generalized linear models (GLM), we will build a multiple linear regression model to predict crop yield based on environmental and management data as input.

Julia: We load a linear regression model provided by [MLJLinearModels.jl](https://github.com/JuliaAI/MLJLinearModels.jl) package.

#### Ex 2.1

Train GLM to predict crop yield in the training data.

In [ ]:
LinearRegressor = @load LinearRegressor pkg = MLJLinearModels

In [ ]:
lr = LinearRegressor()

In [ ]:
lr_mach = machine(lr, X, y)

In [ ]:
fit!(lr_mach, rows = train)

In [ ]:
fitted_params(lr_mach)

#### Ex 2.2

Plot predicted vs. observed yield of training data with a regression line and 1:1 line embedded.

In [ ]:
lr_ŷ_train = predict(lr_mach, X[train, :]);

In [ ]:
scatter(lr_ŷ_train, y[train]; label = "", xlim = (0, 40), ylim = (0, 40), size = (500, 500))
Plots.abline!(1, 0; label = "")

In [ ]:
rmse(lr_ŷ_train, y[train])

#### Ex 2.3

Predict crop yield in the test data based the GLM model you just trained.

In [ ]:
lr_ŷ_test = predict(lr_mach, X[test, :]);

#### Ex 2.4

Plot predicted vs. observed yield of test data with a regression line and 1:1 line embedded.

In [ ]:
scatter(lr_ŷ_test, y[test]; label = "", xlim = (0, 40), ylim = (0, 40), size = (500, 500))
Plots.abline!(1, 0; label = "")

In [ ]:
rmse(lr_ŷ_test, y[test])

#### Ex 2.5

Discuss and interpret your results.

### Random Forest

::: {#exr-ML-RF}
### Random Forest
Construct a random forest regression model to predict crop yield based on environmental and management data as input. For `R`, use \"randomForest\" package.

1.  Train a RF model to predict crop yield in the training data.

2.  Plot predicted vs. observed yield of training data with a regression line and 1:1 line embedded.

3.  Predict crop yield in the test data based the RF model you just trained

4.  Plot predicted vs. observed yield of test data with a regression line and 1:1 line embedded.

5.  Discuss and interpret your results.\
:::


R: Using "randomForest" package, we will a random forest regression model to predict crop yield based on environmental and management data as input.

Julia: We load a random forest model provided by [DecisionTree.jl](https://github.com/JuliaAI/DecisionTree.jl) package.

#### Ex 3.1

Train a RF model to predict crop yield in the training data.

In [ ]:
RandomForestRegressor = @load RandomForestRegressor pkg = DecisionTree

In [ ]:
rf = RandomForestRegressor()

In [ ]:
evaluate(rf, X, y)

In [ ]:
rf_mach = machine(rf, X, y)

In [ ]:
fit!(rf_mach, rows = train)

In [ ]:
rf_mach.model

In [ ]:
fitted_params(rf_mach)

We can also visualize a part of the one of the decision "trees" composing the "forest". Note that decisions are made by comparing "Features" with some fitted values.

In [ ]:
import DecisionTree
DecisionTree.print_tree(fitted_params(rf_mach).forest.trees[1], 5)

"Features" are the variables we supplied for training the model.

In [ ]:
report(rf_mach)

#### Ex 3.2

Plot predicted vs. observed yield of training data with a regression line and 1:1 line embedded.

In [ ]:
rf_ŷ_train = predict(rf_mach, X[train, :]);

In [ ]:
scatter(rf_ŷ_train, y[train]; title="Potato Yield (t/ha)", label = "", xlim = (0, 40), ylim = (0, 40), size = (500, 500))
Plots.abline!(1, 0; label = "")
Plots.xaxis!("Observed (training)")
Plots.yaxis!("RF Predicted (training)")

In [ ]:
rmse(rf_ŷ_train, y[train])

#### Ex 3.3

Predict crop yield in the test data based the RF model you just trained.

In [ ]:
rf_ŷ_test = predict(rf_mach, X[test, :]);

#### Ex 3.4

Plot predicted vs. observed yield of test data with a regression line and 1:1 line embedded.

In [ ]:
scatter(rf_ŷ_test, y[test]; title="Potato Yield (t/ha)",label = "", xlim = (0, 40), ylim = (0, 40), size = (500, 500))
Plots.abline!(1, 0; label = "")
Plots.xaxis!("Observed (testing)")
Plots.yaxis!("RF Predicted (testing)")

In [ ]:
rmse(rf_ŷ_test, y[test])

#### Ex 3.5

Discuss and interpret your results.

### Neural Network

::: {#exr-ML-DNN}
### DNN
Build a DNN regression model to predict crop yield based on environmental and management data as input. Use \"keras\" and \"tensorflow\" packages in R. See the R example code.

1.  Train a DNN model to predict crop yield in the training data.

2.  Plot predicted vs. observed yield of training data with a regression
    line and 1:1 line embedded.

3.  Predict crop yield in the test data based the DNN model you just
    trained

4.  Plot predicted vs. observed yield of test data with a regression
    line and 1:1 line embedded.

5.  Discuss and interpret your results.\
:::


R: Using the data prepared from Ex. 8.1 with "keras" and "tensorflow" packages, we will build a DNN regression model to predict crop yield based on environmental and management data as input.

Julia: We load a neural network regression model provided by [MLJFlux.jl](https://github.com/FluxML/MLJFlux.jl) package, which is built on a popular deep learning package [Flux.jl](https://github.com/FluxML/Flux.jl).

#### Ex 4.1

Train a DNN model to predict crop yield in the training data.

In [ ]:
using Flux

For neural network, data for different variables are put into the same scale.
That is, driving variables should be normalized, or standardized, to scale their effects.

In [ ]:
Standardizer = @load Standardizer pkg = MLJTransforms
# Standardizer = @load Standardizer pkg = MLJModels # v 1.11 or older

In [ ]:
NeuralNetworkRegressor = @load NeuralNetworkRegressor pkg = MLJFlux

Create a DNN model with a few layers of neural network including "hidden" layers. 
The name "hidden" here just means not directly connected to the inputs or outputs.
Two hidden, non-linear, dense layers are added with the [ReLU](https://en.wikipedia.org/wiki/Rectifier_(neural_networks)) function.
Another layer of linear dense single-output is then added.
Define a DNN model to be built and compiled using a default optimizer.

In [ ]:
builder = MLJFlux.@builder Chain(Dense(n_in, 64, relu), Dense(64, 64, relu), Dense(64, n_out))
nn = NeuralNetworkRegressor(; builder, epochs = 200)

The entire model pipeline starts with a standardizer to normalize input data which are fed into the actual neural network model. The output from the model is also transformed via another standardizer to better scale with an actual range of output values.

In [ ]:
pipe = Standardizer |> TransformedTargetModel(nn, transformer = Standardizer)

In [ ]:
nn_mach = machine(pipe, X, y)

In [ ]:
fit!(nn_mach, rows = train)

Plot the history of training. 'epochs' is like iterations of training.

In [ ]:
let L = report(nn_mach).transformed_target_model_deterministic.model.training_losses
    plot(1:length(L), L, label = "", xlab = "Epoch", ylab = "Loss")
end

#### EX 4.2

Plot predicted vs. observed yield of training data with a regression line and 1:1 line embedded.

In [ ]:
nn_ŷ_train = predict(nn_mach, X[train, :]);

In [ ]:
scatter(nn_ŷ_train, y[train]; title="Potato Yield (t/ha)", label = "", xlim = (0, 40), ylim = (0, 40), size = (500, 500))
Plots.abline!(1, 0; label = "")
Plots.xaxis!("Observed (training)")
Plots.yaxis!("DNN Predicted (training)")

In [ ]:
rmse(nn_ŷ_train, y[train])

#### Ex 4.3

Predict crop yield in the test data based the DNN model you just trained.

In [ ]:
nn_ŷ_test = predict(nn_mach, X[test, :]);

#### Ex 4.4

Plot predicted vs. observed yield of test data with a regression line and 1:1 line embedded.

In [ ]:
scatter(nn_ŷ_test, y[test]; title="Potato Yield (t/ha)", label = "", xlim = (0, 40), ylim = (0, 40), size = (500, 500))
Plots.abline!(1, 0; label = "")
Plots.xaxis!("Observed (testing)")
Plots.yaxis!("RF Predicted (testing)")

In [ ]:
rmse(nn_ŷ_test, y[test])

#### Ex 4.5

Discuss and interpret your results.

### Performance evaluation

::: {#exr-ML-evals}
### Performance evaluations
We will do a set of model performance evaluations. In `R`, use \"hydroGOF\" package.

1.  Evaluate and discuss the performance of each modeling method for training data. Compare RMSE, EF (shown as NSE in hydroGOF), and Willmott's $d$ between models. See [@Jeong2016] for their definitions.

2.  Evaluate and discuss the performance of each modeling method for test data. Compare RMSE, EF (shown as NSE in hydroGOF), and Willmott's $d$ between models.

3.  Export the results of training, testing, and performance evaluations to separate .csv files.

4.  Discuss and interpret your results.\
:::

R: We will do a set of model performance evaluations using "hydroGOF" package.

Julia: We instead use internal metric functions from Cropbox.

In [ ]:
import Cropbox

In [ ]:
calculate(O, a...) = begin
    f(n, E) = DataFrame(
        name = n,
        rmse = Cropbox.metricfunc(:rmse)(E, O),
        ef = Cropbox.metricfunc(:ef)(E, O),
        dr = Cropbox.metricfunc(:dr)(E, O),
    )
    vcat([f(n, E) for (n, E) in a]...)
end 

#### Ex 5.1

Evaluate and discuss the performance of each modeling method for training data. Compare RMSE, EF (shown as NSE), and Willmott’s d between models. See Jeong et al. (2016) for their definitions.

In [ ]:
stats_train = calculate(y[train],
    "train.lr" => lr_ŷ_train,
    "train.rf" => rf_ŷ_train,
    "train.nn" => nn_ŷ_train,
)

#### Ex 5.2

Evaluate and discuss the performance of each modeling method for test data. Compare RMSE, EF (shown as NSE), and Willmott’s d between models.

In [ ]:
stats_test = calculate(y[test],
    "test.lr" => lr_ŷ_test,
    "test.rf" => rf_ŷ_test,
    "test.nn" => nn_ŷ_test,
)

#### Ex 5.3

Export the results of training, testing, and performance evaluations to separate .csv files.

In [ ]:
stats = [stats_train; stats_test]

In [ ]:
CSV.write("./data/ML-stats.csv", stats)

#### Ex 5.4

Discuss and interpret your results.

## Homework Problems
### Corn silage production in Northeast Seaboard Region (NESR).
Follow the same modeling process as the potato exercises to create predictions for corn silage data (NESR-silage.csv), evaluate the performance of models, and discuss the results.


In [ ]:
silage = CSV.read("./data/NESR-silage.csv", DataFrame)